In [122]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran

# 자연어 처리 순서
1. 데이터의 로드
2. 데이터 튜닝
3. 데이터 분할
4. 토큰화
5. 벡터화
6. 모델 학습
7. 평가


In [68]:
# 데이터를 로드
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [69]:
# 필요 없는 컬럼 제거 -> id 제외 -> id에 중복 값이 존재하지 않는다.
df.drop(columns='id', axis=1, inplace=True)

In [70]:
# 결측치가 존재하는가?
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  149995 non-null  object
 1   label     150000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.3+ MB


In [71]:
# 결측치가 15만개의 데이터 중 5개의 결측치 관찰
# 5개(굉장히 적은 양) -> 제외
df.dropna(inplace=True)

In [72]:
# 리뷰 데이터 중복된 문장이 존재하는가? -> 확인?
df.value_counts()

document                                                       label
굿                                                              1        165
최고                                                             1         83
쓰레기                                                            0         79
별로                                                             0         66
굳                                                              1         59
                                                                       ... 
""" 그러 자전거 나중에 배우면 되잖아 "" 할때 눈물 팍!!!!! 저 진짜 코까지 풀면서 울었습니다..."  1          1
""" 너에게 감동을 줄테니,너는 나에게 호주머니의 지갑을 열어다오""라고 울부짖고있음"              0          1
""" 아~~~그랬냐~~~발발이 치와와 스치고 왜냐햐면~ 왜냐하면~ """                      1          1
""" 정말 멋진 시에요. 고마워요. """                                       1          1
!                                                              1          1
Name: count, Length: 146339, dtype: int64

In [73]:
# 중복으로 만들어져있는 리뷰 문장들은 제거 -> 과적합 방지
df.drop_duplicates('document', inplace=True)

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146182 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  146182 non-null  object
 1   label     146182 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.3+ MB


In [75]:
X = df['document'].values
Y = df['label'].values

In [76]:
# 학습 데이터와 검증 데이터로 변환
# train test set 으로 분할(분류 데이터 -> label의 비율을 유지)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    stratify=Y,
    random_state=42
)

In [123]:
# X의 데이터가 문자임으로 토큰화 작업
komoran = Komoran()
# 사용할 품사 선택
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
# 사용하지 않을 단어를 선택
stop_word = ['하다', '되다']
# 글자 수 제한
len_word = 2
# 토큰화 함수 정의
def tokenize(text):
    # 결과를 리스트의 되돌려주기 위해 빈 리스트를 생성
    tokens = []
    for word, pos in komoran.pos(text):
        # 조건1 : 품사에 포함되어있다면
        # 조건2 : 금지어에 포함되어있지 않다면
        # 조건3 : 문자의 길이가 len_word보다 크거나 같다면
        if pos in allow_pos and \
        word not in stop_word and \
        len(word) >= len_word:
            #3개의 조건을 모두 만족하는 단어를 tokens 에 추가
            tokens.append(word)
    return tokens

In [78]:
# 벡터 객체 생성 (단어의 중요도를 판단하는 벡터화 class 로드)
# 벡터화 (자연어 데이터에서 사용하는 스케일링 비슷한 작업)
vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1,1),
    min_df= 3, # 3회 이상 나온 단어들을 기준으로 중요도 판단
    lowercase= False
)

In [79]:
# train 데이터를 이용하여 fit_transform()
# test 데이터는 transform() --> 데이터의 누수 방지

In [80]:
X_train_vec = vectorizer.fit_transform(X_train)
print(len(vectorizer.get_feature_names_out()))

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


13575


In [81]:
X_test_vec = vectorizer.transform(X_test)
print(len(vectorizer.get_feature_names_out()))

13575


- train, test 에 모두 fit을 하게 되면 두개의 데이터에서
단어의 추출이 다른 값들을 보인다(데이터의 누수)
- train을 이용하여 fit을 하고 test 데이터는 transform 작업

In [90]:
# X_train_vec.toarray()

In [85]:
# X_train_vec와 Y_train 데이터를 이용하여 모델에 학습
model = LinearSVC(C= 1.0)

In [86]:
# a모델에 학습 -> 13575개의 컬럼에서
# label 0,1 사이의 규칙을 찾아내는 과정
model.fit(X_train_vec, Y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [87]:
pred = model.predict(X_test_vec)

In [88]:
acc = accuracy_score(pred, Y_test)
f1 = f1_score(pred, Y_test)
print(f"정확도 : {round(acc, 4)}, F1score : {round(f1,4)}")

정확도 : 0.7736, F1score : 0.7669


In [114]:
# 모델의 성능을 올리기 위해 최적의 파라미터를 찾는 과정
# Pipeline, GridsearchCV, StratifiedKFold
# 파이프라인 생성
pipe = Pipeline(
    [
        ('vectorizer', TfidfVectorizer(
            # 고정으로 사용할 매개변수의 값 지정
            lowercase=False,
            max_df= 0.95, # 너무 자주 등장하는 단어는 배제
            sublinear_tf=True # tf를 log(1 + tf)로 스케일
        )),
        (
            'clf', LinearSVC()
        )
    ]
) 

In [115]:
# pipe에서 사용할 매개변수의 값들을 지정
param_grid = {
    'vectorizer__ngram_range' : [(1,1),(1,2)],
    'vectorizer__min_df' : [3,5],
    'clf__C' : [0.9, 1.0]
}

In [116]:
# 교차 검증 폴드화
cv = StratifiedKFold(n_splits=3,
                     shuffle=True, random_state=42)

In [117]:
grid = GridSearchCV(
    estimator= pipe,
    param_grid= param_grid,
    scoring= 'f1_macro',
    cv = cv,
    n_jobs= -1,
    verbose= 1
)

In [118]:
# gridsearch을 통한 학습
grid.fit(X_train, Y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


,estimator,Pipeline(step...LinearSVC())])
,param_grid,"{'clf__C': [0.9, 1.0], 'vectorizer__min_df': [3, 5], 'vectorizer__ngram_range': [(1, ...), (1, ...)]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [119]:
print("최적의 매개변수 : ", grid.best_params_)
print("최적의 F1Score : ", round(grid.best_score_,4))

최적의 매개변수 :  {'clf__C': 0.9, 'vectorizer__min_df': 3, 'vectorizer__ngram_range': (1, 2)}
최적의 F1Score :  0.7934


### 실습 문제
- 토큰화
    - komoran 사용
    - 품사는 ('NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL' )만 사용
    - 단어의 길이는 2 이상
    - 금지어 ('하다', '되다')
- 벡터화
    - TfidVectorize 를 사용
    - 고정 매개변수는
        - tokenizer = komoran 사용
        - lowercase = False
        - max_df = 0.95
        - sublinear_tf = True
- 분류 모델
    - LogisticRegression
    - 고정 매개변수
        - n_jobs = -1
        - class_weight = 'balanced'
        
- 파이프라인 생성
    - 벡터화
    - 분류 모델


- 교차 검증
    - StratifiedKFold
        - 폴드의 개수를 4
        - shuffle : True
        - random_state = 42

- gridsearch
    - 벡터화
        - ngram_range 는 (1, 1), (1, 2)
        - min_df 는 3, 5
    - 분류 모델
        - max_iter를 800, 1000
        - C 를 1.0, 2.0 
- 최적의 파라미터를 찾는다

In [148]:
# X의 데이터가 문자임으로 토큰화 작업
komoran = Komoran()
# 사용할 품사 선택
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
# 사용하지 않을 단어를 선택
stop_word = ['하다', '되다']
# 글자 수 제한
len_word = 2
# 토큰화 함수 정의
def tokenize(text):
    # 결과를 리스트의 되돌려주기 위해 빈 리스트를 생성
    tokens = []
    for word, pos in komoran.pos(text):
        # 조건1 : 품사에 포함되어있다면
        # 조건2 : 금지어에 포함되어있지 않다면
        # 조건3 : 문자의 길이가 len_word보다 크거나 같다면
        if pos in allow_pos and \
        word not in stop_word and \
        len(word) >= len_word:
            #3개의 조건을 모두 만족하는 단어를 tokens 에 추가
            tokens.append(word)
    return tokens

In [149]:
from sklearn.linear_model import LogisticRegression

In [150]:
model2 = LogisticRegression(
    n_jobs= -1,
    class_weight='balanced'
)

In [151]:
model2.fit(X_train_vec, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [170]:
pipe = Pipeline(
    [
        ('vectorizer', TfidfVectorizer(
            # 고정으로 사용할 매개변수의 값 지정
            lowercase=False,
            max_df= 0.95, # 너무 자주 등장하는 단어는 배제
            sublinear_tf=True # tf를 log(1 + tf)로 스케일
        )),
        (
            'clf', model2
        )
    ]
) 

In [171]:
# pipe에서 사용할 매개변수의 값들을 지정
param_grid = {
    'vectorizer__ngram_range' : [(1,1),(1,2)],
    'vectorizer__min_df' : [3,5],
    'clf__C' : [1.0, 2.0],
    'clf__max_iter' : [800, 1000]
}

In [172]:
# 교차 검증 폴드화
cv = StratifiedKFold(n_splits=4,
                     shuffle=True, random_state=42)

In [173]:
grid = GridSearchCV(
    estimator= pipe,
    param_grid= param_grid,
    scoring= 'f1_macro',
    cv = cv,
    n_jobs= -1,
    verbose= 1
)

In [174]:
grid.fit(X_train, Y_train)

Fitting 4 folds for each of 16 candidates, totalling 64 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

In [163]:
print("최적의 매개변수 : ", grid.best_params_)
print("최적의 F1Score : ", round(grid.best_score_,4))

최적의 매개변수 :  {'clf__C': 2.0, 'clf__max_iter': 800, 'vectorizer__min_df': 3, 'vectorizer__ngram_range': (1, 2)}
최적의 F1Score :  0.8024
